# 002: Understanding the Framework Layer

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from earlysign.core.ledger import Ledger
import ibis

connection = ibis.connect("duckdb://:memory:")
ledger_bare = Ledger(connection, "example_bare").bind(exp_id="exp_001")
ledger_struct = Ledger(connection, "example_structured").bind(exp_id="exp_001")

ledger_bare.ensure()
ledger_struct.ensure()

ledger_bare, ledger_struct

(Ledger(connector=<ibis.backends.duckdb.Backend object at 0x10ecee590>, table_name='example_bare', labels={'exp_id': 'exp_001'}),
 Ledger(connector=<ibis.backends.duckdb.Backend object at 0x10ecee590>, table_name='example_structured', labels={'exp_id': 'exp_001'}))

## LedgerRecord

If we do this directly using the ledger syntax:

In [3]:
ledger_bare.insert(
    "",
    {"nA": 100, "nB": 100, "mA": 10, "mB": 35},
    {"record_id": "example_data_protocol"},
)
ledger_bare.show()

,payload_name,payload,labels
0,,"{'mA': 10, 'mB': 35, 'nA': 100, 'nB': 100}","{'exp_id': 'exp_001', 'record_id': 'example_da..."


Instead, we can use the framework layer:

In [4]:
from earlysign.v0.framework.records import LedgerRecord, QueryMixin


class Data(LedgerRecord, QueryMixin):
    schema = {"nA": int, "nB": int, "mA": int, "mB": int}


# Instantiate and attach to the ledger
data_adapter = Data(name="example_data_protocol")
data_adapter.attach(ledger_struct)

# Insert a new record
data_adapter.insert(nA=100, nB=100, mA=10, mB=35)

ledger_struct.show()

,payload_name,payload,labels
0,Data,"{'mA': 10, 'mB': 35, 'nA': 100, 'nB': 100}","{'exp_id': 'exp_001', 'record_name': 'example_..."


Now, suppose we want to read these records. Let's say we have some more data written in the ledger by other operations.

In [5]:
ledger_bare.insert("", {"some": "other", "data": "by other operations"})
ledger_bare.insert("", {"some": "other", "data": "by other operations"})
ledger_bare.insert("", {"some": "other", "data": "by other operations"})
ledger_bare.show()

,payload_name,payload,labels
0,,"{'mA': 10, 'mB': 35, 'nA': 100, 'nB': 100}","{'exp_id': 'exp_001', 'record_id': 'example_da..."
1,,"{'data': 'by other operations', 'some': 'other'}",{'exp_id': 'exp_001'}
2,,"{'data': 'by other operations', 'some': 'other'}",{'exp_id': 'exp_001'}
3,,"{'data': 'by other operations', 'some': 'other'}",{'exp_id': 'exp_001'}


In [6]:
ledger_struct.insert("", {"some": "other", "data": "by other operations"})
ledger_struct.insert("", {"some": "other", "data": "by other operations"})
ledger_struct.insert("", {"some": "other", "data": "by other operations"})
ledger_struct.show()

,payload_name,payload,labels
0,Data,"{'mA': 10, 'mB': 35, 'nA': 100, 'nB': 100}","{'exp_id': 'exp_001', 'record_name': 'example_..."
1,,"{'data': 'by other operations', 'some': 'other'}",{'exp_id': 'exp_001'}
2,,"{'data': 'by other operations', 'some': 'other'}",{'exp_id': 'exp_001'}
3,,"{'data': 'by other operations', 'some': 'other'}",{'exp_id': 'exp_001'}


Then, in order to read out the records using only the bare ledger functionality, you need to carefully write the logic to query the relevant data record based on the label values.

In [7]:
ledger_bare.t.filter(
    ledger_bare.t.labels["record_id"].str == "example_data_protocol"
).execute()
# type(ledger_bare.t.select(ledger_bare.t.labels["record_id"]).execute().to_dict()["JSONGetItem(labels, 'record_id')"][0])
# print('ledger_bare.t.labels["record_id"].cast("string").execute()')
# print(ledger_bare.t.labels["record_id"].cast("string").execute())
# print('ledger_bare.t.labels["record_id"].str.execute()')
# print(ledger_bare.t.labels["record_id"].str.execute())

,uuid,ts,pkg_version,payload_type,payload,labels
0,73a229949d9e42aaa737fddd3d239aab,2025-11-14 04:21:56.626413+00:00,earlysign==0.2.0.post174,,"{'mA': 10, 'mB': 35, 'nA': 100, 'nB': 100}","{'exp_id': 'exp_001', 'record_id': 'example_da..."


On the other hand, if you are using the framework layer, the same can be achieved by simple calls of the helper methods without specifying the query on your own; the LedgerRecord subclass encapsulates the boilerplate and does the work for you.
Compare this code against the above.

In [8]:
data_adapter.latest().execute()

,uuid,ts,pkg_version,payload_type,payload,labels,nA,nB,mA,mB
0,13274f6743514ae9a11ad8e081932064,2025-11-14 04:21:56.859614+00:00,earlysign==0.2.0.post174,__main__.Data,"{'mA': 10, 'mB': 35, 'nA': 100, 'nB': 100}","{'exp_id': 'exp_001', 'record_name': 'example_...",100,100,10,35


This adds another layer of abstraction that comes in handy in more complex scenarios,
where complex calculation operations can focus on the logic itself, not on how to read out the appropriate ledger record from the history or other boilerplate.

## LedgerOp

LedgerOps take one or more LedgerRecords at the time of instantiation,
reads the contents of the ledger through the LedgerRecord instances,
and writes new events to the ledger through the LedgerRecord instances.

Conceptually, they represent operations on the ledger.
They perform the read/write actions only through the LedgerRecords.

In [9]:
from earlysign.v0.framework.operator import LedgerOp


class MyCalculation(LedgerOp):
    def __init__(self, data_adapter: Data):
        self.data_adapter = data_adapter

    def run(self):
        self.data_adapter.latest().select("nA").execute().squeeze()


op = MyCalculation(data_adapter)
op.run()

There are various off-the-shelf records and operators.